### INSTALAÇÕES - SE NECESSÁRIO -




In [ ]:
# Instalar pacotes necessários (se ainda não instalados)
!pip install nltk scikit-learn

In [ ]:
!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.5 MB/s eta 0:00:00


### IMPORTAÇÕES

In [ ]:
import glob
import json
import pandas as pd
import re
import unidecode
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import os

In [ ]:
# prompt: conecte ao drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### TRATAMENTO DOS DADOS

In [ ]:
# Configurar NLTK (stopwords e stemmer)

# Baixar recursos do NLTK
nltk.download('stopwords')
nltk.download('rslp')

# Configurações
stopwords_pt = set(stopwords.words('portuguese'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


Opção 2 de pré processamento

In [ ]:
# Lista de stopwords padrão do português
stopwords_pt = set(stopwords.words('portuguese'))

# Stemmer
stemmer = RSLPStemmer()

# Lista de palavras-chave (cidades e candidatos) a serem excluídas
palavras_excluir = {
    'aracaju', 'emilia', 'correa', 'luiz', 'roberto',
    'fortaleza', 'andre', 'fernandes', 'evandro', 'leitao',
    'joao', 'pessoa', 'cicero', 'lucena', 'luciano', 'cartaxo',
    'maceio', 'jhc', 'rafael', 'brito',
    'natal', 'natalia', 'bonavides', 'paulinho', 'freire',
    'recife', 'gilson', 'machado', 'neto', 'campos',
    'salvador', 'bruno', 'reis', 'geraldo', 'jr', 'kleber', 'rosa',
    'sao', 'luis', 'duarte', 'eduardo', 'braide',
    'teresina', 'fabio', 'novo', 'silvio', 'mendes',  'marcelo', 'queiroga', 'propaganda', 'eleitoral'
}

def preprocessar_texto(texto):
    if pd.isna(texto) or texto.strip() == '':
        return ''

    texto = texto.lower()

    # Remover URLs
    texto = re.sub(r"http\S+|www\S+", '', texto)

    # Remover hashtags e menções
    texto = re.sub(r'#\S+', '', texto)
    texto = re.sub(r'@\S+', '', texto)

    # Remover caracteres não alfabéticos
    texto = re.sub(r'[^a-zA-ZÀ-ÿ\s]', '', texto)

    # Remover acentuação
    texto = unidecode.unidecode(texto)

    # Tokenização
    palavras = texto.split()

    # Remover stopwords e palavras a excluir
    palavras_filtradas = [p for p in palavras if p not in stopwords_pt and p not in palavras_excluir]

    return ' '.join(palavras_filtradas)


## Centro

### TRATAMENTO

In [ ]:
# Carregar todos os JSONs da pasta
caminho_json = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/eixo/centro/*.json"
arquivos = glob.glob(caminho_json)

# Lista para armazenar os textos
todos_os_textos = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)
        todos_os_textos.extend(df['texto_processado'].dropna().tolist())


Remoção de termos que aparecem com muita frequência (>=90%) ou pouca frequência (<=10%)

In [ ]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(todos_os_textos)

# Obter termos filtrados
termos_filtrados = vectorizer.get_feature_names_out()

### WORD CLOUDS

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Centro", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/centro.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_centro.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")


## Direita

### TRATAMENTO

In [ ]:
# Carregar todos os JSONs da pasta
caminho_json = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/eixo/direita/*.json"
arquivos = glob.glob(caminho_json)

# Lista para armazenar os textos
todos_os_textos = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)
        todos_os_textos.extend(df['texto_processado'].dropna().tolist())


In [ ]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(todos_os_textos)

# Obter termos filtrados
termos_filtrados = vectorizer.get_feature_names_out()


### WORDCLOUDS

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Direita", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/direita.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_direita.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")


## Esquerda

### TRATAMENTO

In [ ]:
# Carregar todos os JSONs da pasta
caminho_json = "/content/drive/My Drive/Colab Notebooks/ENIAC/2.0/eixo/esquerda/*.json"
arquivos = glob.glob(caminho_json)

# Lista para armazenar os textos
todos_os_textos = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        posts = json.load(f)
        df = pd.DataFrame(posts)
        df['texto_processado'] = df['caption'].apply(preprocessar_texto)
        todos_os_textos.extend(df['texto_processado'].dropna().tolist())


In [ ]:
# Aplicar TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = vectorizer.fit_transform(todos_os_textos)

# Obter termos filtrados
termos_filtrados = vectorizer.get_feature_names_out()


### WORDCLOUDS

In [ ]:
os.makedirs("wordclouds", exist_ok=True)

# Somar os pesos TF-IDF por termo (across all documents)
somas_tfidf = X_tfidf.sum(axis=0).A1
pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

# Criar WordCloud
wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='Dark2').generate_from_frequencies(pesos_por_termo)

# Mostrar
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("WordCloud - Esquerda", fontsize=18)
# Substituir espaços ou caracteres inválidos no nome do arquivo
nome_arquivo = f"wordclouds/esquerda.png"
plt.savefig(nome_arquivo, bbox_inches='tight')
plt.close()  # Fecha a figura para economizar memória


In [ ]:
# Imprimir os 10 termos mais frequentes
# Abre (ou cria) o arquivo txt para escrita
with open("top_10_termos_esquerda.txt", "w", encoding="utf-8") as f:
  # Já temos: pesos_por_termo = dict(zip(termos_filtrados, somas_tfidf))

  # Ordenar os termos por peso TF-IDF (do maior para o menor)
  termos_ordenados = sorted(pesos_por_termo.items(), key=lambda x: x[1], reverse=True)

  # Imprimir os 10 primeiros
  f.write("Top 10 termos mais frequentes:\n")
  for termo, peso in termos_ordenados[:10]:
      f.write(f"{termo} ({peso:.3f})\n")
